# 02 – Configure

This notebook uses the **Python SDK** to verify the deployed Azure OpenAI model is ready and confirm the deployment name before running prompts.

For a simple model deployment like this one there is no index or data pipeline to set up — the configure step is a readiness check. For more complex demos (e.g. RAG with AI Search), this notebook would create indexes and upload documents.

**Pre-requisite:** `01_deploy_infra.ipynb` must have run and written `env/.env`.

---

## Step 1 – Load environment variables

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

endpoint    = os.environ["AZURE_OPENAI_ENDPOINT"]
api_key     = os.environ["AZURE_OPENAI_API_KEY"]
deployment  = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "chat")
api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21")

print(f"Endpoint   : {endpoint}")
print(f"Deployment : {deployment}")
print(f"API version: {api_version}")

## Step 2 – Create the Azure OpenAI client

In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=api_key,
    api_version=api_version,
)
print("✅  Client created")

## Step 3 – Verify the deployment is ready

List deployments on the account and confirm ours is present and in a `succeeded` state.

In [ ]:
import subprocess, json

result = subprocess.run(
    ["az", "cognitiveservices", "account", "deployment", "list",
     "--name", os.environ.get("AZURE_OPENAI_ACCOUNT_NAME", deployment),
     "--resource-group", os.environ.get("AZURE_RESOURCE_GROUP", ""),
     "--query", "[].{name:name, state:properties.provisioningState, model:properties.model.name}",
     "--output", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    deployments = json.loads(result.stdout)
    for d in deployments:
        status = "✅" if d.get("state") == "Succeeded" else "⚠️"
        print(f"{status}  {d['name']} ({d['model']}) — {d['state']}")
else:
    # If the az query fails (e.g. account name not in env), do a lightweight API ping instead
    print("Could not query deployments via CLI — trying a lightweight API ping...")
    response = client.chat.completions.create(
        model=deployment,
        messages=[{"role": "user", "content": "ping"}],
        max_tokens=1,
    )
    print(f"✅  Deployment '{deployment}' responded")

---

## ✅ Deployment verified

The model is reachable and ready. Move to **`03_test.ipynb`** to run prompts.